# 02 — Ipotesi 1: livello di vol con soglia

BEAR se la vol supera il suo percentile q80 (rolling, solo dati passati). È la "colonna Fear": stress = vol alta.

In [1]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import vol_regime as vr
pd.set_option("display.float_format", lambda x: f"{x:.3f}")

vol = vr.load_vol()

## Hit ratio: il segnale predice la direzione del mese successivo? (vs proxy)

In [2]:
for ac, vcol in vr.VOL_FOR_CLASS.items():
    v = vol[vcol].dropna()
    sig = vr.signal_threshold(v, q=0.80)
    print(f"==== {ac} (vol={vcol}) ====")
    print(vr.compare_table(sig, ac).to_string(float_format=lambda x: f"{x:.3f}"))
    print()

==== Equity (vol=VIX) ====
                hit_ratio       n  base_rate_up  bull_share
Equity (h=21d)                                             
signal              0.562 317.000         0.618       0.779
proxy               0.615 317.000         0.618       0.700

==== Fixed Income (vol=MOVE) ====
                      hit_ratio       n  base_rate_up  bull_share
Fixed Income (h=21d)                                             
signal                    0.522 270.000         0.511       0.811
proxy                     0.510 286.000         0.514       0.493



## Stabilità: hit ratio per anno (Equity)

In [3]:
v = vol["VIX"].dropna()
sig = vr.signal_threshold(v, q=0.80)
px = vr.load_prices()["SPY"].dropna()
sig_al = sig.reindex(px.index).ffill()
vr.hit_ratio_by_year(sig_al, px).round(3)

2000   0.333
2001   0.083
2002   0.417
2003   0.750
2004   0.583
2005   0.750
2006   0.667
2007   0.417
2008   0.500
2009   0.750
2010   0.583
2011   0.583
2012   0.583
2013   0.917
2014   0.500
2015   0.417
2016   0.583
2017   0.917
2018   0.583
2019   0.583
2020   0.333
2021   0.667
2022   0.167
2023   0.750
2024   0.750
2025   0.500
2026   0.400
Name: correct, dtype: float64

**Conclusione (02).** Il segnale a soglia **non batte il proxy** sull'hit ratio
azionario (≈0.56 vs ≈0.62, sotto anche il base-rate 0.62): la VIX alta è
*coincidente* col drawdown, quindi la soglia entra in BEAR a crash già iniziato.
Su Fixed Income tutti ~0.51 (coin flip).